In [ ]:
# ============================================================
# RetailPulse | Phase 1: Data Inspection
# Author: Naisha
# Date: May 2026
# Purpose: Load all tables, inspect shape, nulls, and dtypes
# ============================================================

import pandas as pd
import os

# ── 1. Load all tables ──────────────────────────────────────
RAW = "../data/raw/"

tables = {
    "customers"   : "olist_customers_dataset.csv",
    "orders"      : "olist_orders_dataset.csv",
    "order_items" : "olist_order_items_dataset.csv",
    "payments"    : "olist_order_payments_dataset.csv",
    "products"    : "olist_products_dataset.csv",
    "sellers"     : "olist_sellers_dataset.csv",
    "reviews"     : "olist_order_reviews_dataset.csv",
    "geolocation" : "olist_geolocation_dataset.csv",
    "categories"  : "product_category_name_translation.csv"
}

dfs = {}
for name, file in tables.items():
    dfs[name] = pd.read_csv(os.path.join(RAW, file))
    print(f"✓ Loaded {name:15s} → {dfs[name].shape}")

In [ ]:
# ── 2. Health report on every table ─────────────────────────
for name, df in dfs.items():
    print(f"\n{'='*55}")
    print(f" TABLE: {name.upper()}")
    print(f"{'='*55}")
    print(f" Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")
    print(f"\n Null counts:")
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls) == 0:
        print("  → No nulls found ✓")
    else:
        for col, count in nulls.items():
            pct = count / len(df) * 100
            print(f"  → {col}: {count:,} nulls ({pct:.1f}%)")
    print(f"\n Column types:")
    for col, dtype in df.dtypes.items():
        print(f"  → {col}: {dtype}")
        
        # ── 3. Preview each table ────────────────────────────────────
for name, df in dfs.items():
    print(f"\n{'='*55}")
    print(f" TABLE: {name.upper()} — first 3 rows")
    print(f"{'='*55}")
    display(df.head(3))

In [ ]:
dfs['customers'].head()


In [ ]:
dfs['customers']['customer_state'].value_counts()

## Customers Table — First Observations

- 5 columns: two IDs, zip code, city, state
- customer_unique_id is the true loyalty identifier
- 42% of customers are from São Paulo (SP)
- Top 3 states cover 67% of all customers
- 14+ states are significantly underserved

In [ ]:
dfs['orders'].head()

In [ ]:
dfs['orders']['order_status'].value_counts()

## Orders Table — First Observations

- 8 columns including purchase, approval, and delivery timestamps
- 97% of orders were successfully delivered
- 625 cancelled orders — needs investigation in cleaning phase
- Can calculate delivery delay = actual vs estimated delivery date

In [ ]:
dfs['payments'].head()

In [ ]:
dfs['payments']['payment_type'].value_counts()

In [ ]:
dfs['payments']['payment_installments'].describe()

## Payments Table — First Observations

- payment_value is the core revenue column
- 74% of orders paid by credit card
- 50% of customers pay in full (1 installment)
- Max installments = 24 months — high value purchases
- 3 rows with payment_type = not_defined → drop in cleaning
- Min installments = 0 → data error, fix in cleaning

## Null Report — Key Findings

### Issues to fix in cleaning:
1. Orders: 5 date columns loaded as object → convert to datetime
2. Orders: delivery nulls expected (cancelled/in-transit orders)
3. Products: 610 nulls in category → fill with "unknown"
4. Products: typo in column name → rename to product_name_length
5. Payments: 3 rows with payment_type = not_defined → drop
6. Payments: rows with 0 installments → investigate and drop

### Tables that are clean:
- customers, order_items, sellers, geolocation, categories